In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
default_of_credit_card_clients = fetch_ucirepo(id=350) 
  
# data (as pandas dataframes) 
X = default_of_credit_card_clients.data.features 
y = default_of_credit_card_clients.data.targets 
  
# metadata 
print(default_of_credit_card_clients.metadata) 
  
# variable information 
print(default_of_credit_card_clients.variables) 

df = pd.concat([X, y], axis=1)

{'uci_id': 350, 'name': 'Default of Credit Card Clients', 'repository_url': 'https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients', 'data_url': 'https://archive.ics.uci.edu/static/public/350/data.csv', 'abstract': "This research aimed at the case of customers' default payments in Taiwan and compares the predictive accuracy of probability of default among six data mining methods.", 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 30000, 'num_features': 23, 'feature_types': ['Integer', 'Real'], 'demographics': ['Sex', 'Education Level', 'Marital Status', 'Age'], 'target_col': ['Y'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Fri Mar 29 2024', 'dataset_doi': '10.24432/C55S3H', 'creators': ['I-Cheng Yeh'], 'intro_paper': {'ID': 365, 'type': 'NATIVE', 'title': 'The comparisons of data mining techniques for the predictive accuracy of 

In [11]:
# Data pipiline


import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

data = df.copy()
data.columns = data.columns.str.strip()

# Detect target column
target = next(
    (c for c in [
        "Y",
        "default.payment.next.month",
        "default payment next month"
    ] if c in data.columns),
    None
)

if target is None:
    raise ValueError("Target column not found.")

# Basic inspection
print("Missing values:")
print(data.isna().sum())

print("\nDuplicate IDs:")
if "ID" in data.columns:
    print(data["ID"].duplicated().sum())

print("\nTarget distribution:")
print(data[target].value_counts(normalize=True))

# Remove rows with invalid or missing target labels
data = data[data[target].isin([0, 1])].copy()

y = data[target].astype(int)
X = data.drop(columns=[target, "ID"], errors="ignore")

pay_cols = [f"X{i}" for i in range(6, 12)]
bill_cols = [f"X{i}" for i in range(12, 18)]
payment_cols = [f"X{i}" for i in range(18, 24)]

def engineer_features(X):
    X = X.copy()

    # Age: mark implausible values as missing
    X["X5"] = X["X5"].where(
        X["X5"].between(18, 100), np.nan
    )

    # Invalid credit limits
    X["X1"] = X["X1"].where(X["X1"] > 0, np.nan)

    # Gender: preserve valid categories
    X["X2"] = X["X2"].where(
        X["X2"].isin([1, 2]), np.nan
    )

    # Education: 0 is reserved for unknown
    X["X3"] = X["X3"].where(
        X["X3"].isin([1, 2, 3, 4]), 0
    )

    # Marriage: 0 is reserved for unknown
    X["X4"] = X["X4"].where(
        X["X4"].isin([1, 2, 3]), 0
    )

    # Repayment behavior
    delays = X[pay_cols].clip(lower=0)

    X["delinquent_months"] = (
        X[pay_cols] > 0
    ).sum(axis=1)

    X["max_delay"] = delays.max(axis=1)

    X["severe_delay_months"] = (
        X[pay_cols] >= 2
    ).sum(axis=1)

    # Bill features
    X["avg_bill"] = X[bill_cols].mean(axis=1)

    X["recent_bill_change"] = (
        X["X12"] - X["X13"]
    )

    # Payment features
    X["avg_payment"] = X[payment_cols].mean(axis=1)

    X["zero_payment_months"] = (
        X[payment_cols] == 0
    ).sum(axis=1)

    # Credit utilization proxy
    X["credit_utilization"] = (
        X["X12"].clip(lower=0) / X["X1"]
    )

    # Payment-to-bill proxy
    positive_bills = X[bill_cols].clip(lower=0)
    total_bills = positive_bills.sum(axis=1)
    total_payments = X[payment_cols].sum(axis=1)

    X["payment_to_bill_ratio"] = (
        total_payments /
        total_bills.replace(0, np.nan)
    ).clip(lower=0, upper=5)

    return X


Missing values:
X1     0
X2     0
X3     0
X4     0
X5     0
X6     0
X7     0
X8     0
X9     0
X10    0
X11    0
X12    0
X13    0
X14    0
X15    0
X16    0
X17    0
X18    0
X19    0
X20    0
X21    0
X22    0
X23    0
Y      0
dtype: int64

Duplicate IDs:

Target distribution:
Y
0    0.7788
1    0.2212
Name: proportion, dtype: float64


In [12]:

# 60% train, 20% validation, 20% test
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev,
    test_size=0.25,
    stratify=y_dev,
    random_state=42
)

# Deterministic row-level engineering
X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Train: (18000, 32)
Validation: (6000, 32)
Test: (6000, 32)


In [13]:

categorical_cols = [
    "X2", "X3", "X4"
] + pay_cols

numerical_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", RobustScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])


In [14]:

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        random_state=42
    ),
    "Linear SVC": LinearSVC(
        max_iter=10000,
        random_state=42
    )
}

fitted_models = {}

for name, estimator in models.items():

    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", estimator)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_val)

    if hasattr(pipeline, "predict_proba"):
        y_score = pipeline.predict_proba(X_val)[:, 1]
    else:
        y_score = pipeline.decision_function(X_val)

    print(f"\n{name}")
    print(classification_report(
        y_val, y_pred, zero_division=0
    ))

    print("ROC-AUC:", roc_auc_score(y_val, y_score))
    print("PR-AUC (AP):", average_precision_score(
        y_val, y_score
    ))

    print("Confusion matrix:")
    print(confusion_matrix(y_val, y_pred))

    fitted_models[name] = pipeline



Logistic Regression
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      4673
           1       0.69      0.35      0.47      1327

    accuracy                           0.82      6000
   macro avg       0.76      0.65      0.68      6000
weighted avg       0.80      0.82      0.80      6000

ROC-AUC: 0.7754516115038838
PR-AUC (AP): 0.5452047936779886
Confusion matrix:
[[4459  214]
 [ 860  467]]

Linear SVC
              precision    recall  f1-score   support

           0       0.84      0.96      0.89      4673
           1       0.68      0.34      0.46      1327

    accuracy                           0.82      6000
   macro avg       0.76      0.65      0.67      6000
weighted avg       0.80      0.82      0.80      6000

ROC-AUC: 0.7733613274223113
PR-AUC (AP): 0.5430685823415591
Confusion matrix:
[[4463  210]
 [ 872  455]]
